# Suite BLW — A store given a store below

A store may be given the store below it, from which it loads what it is asked for, when it is asked ([CAP-283: a store given a store below](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-283), [CAP-284: objects loaded on demand](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-284)): `Builders.Store(below)`. The store below gives an object's document, by its path (`document(path)`): a one-transaction document of its properties and its entries, each object they link by its path. The store above loads an object the first time its values or entries are read, as its committed data, observed by none; an object is present in it, made in it or loaded into it, or it is not: an entry that links one not present holds it by its path, which reading the link loads, and a path the store below holds no object at resolves to none. Given no store below, a store holds what is written to it, as it always has. A transaction of the store above is written through, as plain data, before it commits: the store below commits it first, and a refusal there refuses it; and what the store below commits, the store above applies to the objects present in it alone, in the order the store below committed it, loading nothing ([CAP-285: writes through, others' writes in](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-285)).

Python and TypeScript run this suite with the same case IDs, titles and asserted values. The case setup is below.

In [ ]:
import { Builders as B, Changes, JSON as Json, Plain, Proxies, References, Repr } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, raises } from "./support.js";

/** crm.Person, owned by the singleton book by key, with a name and a home; persons know persons; a book has one lead;
 * persons mentor persons; persons are tagged with counted tags. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Home: S.OfObject, Holds: S.OfRelation, Knows: S.OfRelation, Book: S.OfObject,
      Person: S.OfObject, Keeps: S.OfRelation, Pet: S.OfObject, Walks: S.OfRelation, Introduces: S.OfRelation,
      Leads: S.OfRelation, Mentors: S.OfRelation, Tag: S.OfObject, Tagged: S.OfRelation })
    .spec([
      (_: any) => _.Home.name("crm.Home").properties({ city: S.OfStr, zip: S.OfStr }),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")])
        .properties({ key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]),
      (_: any) => _.Keeps.name("crm.Keeps").links([(t: any) => t.name("keeper"), (t: any) => t.name("pet")])
        .properties({ name: S.OfStr }).uniques([["keeper", "name"], ["pet"]]),
      (_: any) => _.Introduces.name("crm.Introduces").links([(t: any) => t.name("host"), (t: any) => t.name("guest"),
        (t: any) => t.name("friend")]),
      (_: any) => _.Leads.name("crm.Leads").links([(t: any) => t.name("book"), (t: any) => t.name("lead")])
        .uniques([["lead"]]), // a book's one lead
      (_: any) => _.Mentors.name("crm.Mentors").links([(t: any) => t.name("mentor"), (t: any) => t.name("mentee")]),
      (_: any) => _.Tagged.name("crm.Tagged").links([(t: any) => t.name("tagged"), (t: any) => t.name("tag")]),
      (_: any) => _.Tag.name("crm.Tag").ref(true).properties({ label: S.OfStr }).adjacencies({ // counted
        tagged: (t: any) => t.relation([_.Tagged]).me("tag") }),
      (_: any) => _.Walks.name("crm.Walks").links([(t: any) => t.name("walker"), (t: any) => t.name("walked")]),
      (_: any) => _.Pet.name("crm.Pet").ref(true).properties({ kind: S.OfStr }).adjacencies([
        (t: any) => t.name("keeper").relation([_.Keeps]).me("pet").owner(true),
        (t: any) => t.name("walkers").relation([_.Walks]).me("walked")]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book"),
        leads: (t: any) => t.relation([_.Leads]).me("book") }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr, home: _.Home }).adjacencies([
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true),
        (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
        (t: any) => t.name("known_by").relation([_.Knows]).me("known"),
        (t: any) => t.name("pets").relation([_.Keeps]).me("keeper"),
        (t: any) => t.name("walks").relation([_.Walks]).me("walker"),
        (t: any) => t.name("mentors").relation([_.Mentors]).me("mentor"), // a mentee's side has no adjacency
        (t: any) => t.name("tags").relation([_.Tagged]).me("tagged"),
        (t: any) => t.name("hosts").relation([_.Introduces]).me("host"),
        (t: any) => t.name("guested").relation([_.Introduces]).me("guest"),
        (t: any) => t.name("befriended").relation([_.Introduces]).me("friend"),
      ]),
    ])]))]);
}

const below: any = new B.Store();
below.transaction(crm);
const book = below.singleton("crm.TheBook");
const people: Record<string, any> = {};
below.transaction(() => {
  for (const n of ["ada", "bob", "cy"]) {
    people[n] = below.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key(n).book(book))).name(n));
  }
  below.crm.Person(people.ada, (p: any) => p.home((h: any) => h.city("Paris").zip("75001"))
    .knows((a: any) => a.insert((e: any) => e.known(people.bob))));
  const rexBelow = below.crm.Pet((p: any) => p.keeper((a: any) => a.insert((e: any) => e.name("rex").keeper(people.ada)))
    .kind("dog"));
  below.crm.Person(people.bob, (p: any) => p.walks((a: any) => a.insert((e: any) => e.walked(rexBelow)))
    .hosts((a: any) => a.insert((e: any) => e.guest(people.ada).friend(people.cy))));
});
const above: any = new B.Store(below);
above.transaction(crm);

/** How many objects of the store below's a store above has loaded, an int. */
function loaded(store: any = above): bigint {
  return BigInt(store._loaded.size);
}

/** The entry of an adjacency whose key is `key`. */
function keyed(adjacency: any, key: string): any {
  return [...adjacency].find((e: any) => e.key.get() === key);
}

/** Plain data: a path written as an object literal, as `Map`s. */
function plain(literal: unknown): any {
  return Json.loads(JSON.stringify(literal));
}

<a id="BLW-01"></a>
## BLW-01 · Nothing is loaded until it is read; reading an object loads it alone, an entry linking one not yet loaded holding it by its path until the link is read

In [ ]:
const ours = above.singleton("crm.TheBook");
const before = loaded();
const keys = [...ours.contacts].map((e: any) => e.key.get());
const afterBook = loaded();
const ada = keyed(ours.contacts, "ada").contact;
const name = ada.name.get();
assert(Json.dumps([before, keys, afterBook, name, loaded()]) === '[0, ["ada", "bob", "cy"], 1, "ada", 2]');

<a id="BLW-02"></a>
## BLW-02 · An object is the same object each time it is named, through any entry; its value objects and entries are loaded with it; an object reached by its reference, or through an entry, loads alone, its owner not loaded, its reference through its owner's path

In [ ]:
const bob = keyed(ours.contacts, "bob").contact;
assert([...ada.knows][0].known === bob && [...ada.book][0].book === ours);
assert([...bob.known_by][0].knower === ada);
assert(Json.dumps([ada.home.city.get(), ada.home.zip.get(), bob.name.get(), ada.as_schema().name]) === '["Paris", "75001", "bob", "crm.Person"]');
const fresh: any = new B.Store(below);
fresh.transaction(crm);
const rexPath = plain({ singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book", properties: { key: "ada" },
  next: { entry: { i: "pets", o: "keeper", properties: { name: "rex" }, next: { target: {} } } } } } });
const rex: any = References.resolve(fresh, Plain.FromPlain(fresh)(fresh.registered("References.OfObject"), new Map([["path", rexPath]])));
const rexReference = Json.dumps(Plain.ToPlain(fresh)(fresh.registered("References.OfObject"), References.of(fresh, rex)));
assert(rexReference === Json.dumps(new Map([["path", rexPath]]))); // its owner not present
assert(Json.dumps([loaded(fresh), Repr.repr(rex), rex.kind.get(), loaded(fresh)]) === '[1, "<crm.Pet>", "dog", 1]');
const other: any = new B.Store(below);
other.transaction(crm);
const walker = keyed(other.singleton("crm.TheBook").contacts, "bob").contact;
const walked = [...walker.walks][0].walked;
assert(Json.dumps([loaded(other), Repr.repr(walked), walked.kind.get(), loaded(other)]) === '[3, "<crm.Pet>", "dog", 3]');

<a id="BLW-03"></a>
## BLW-03 · The store below gives an object's document by its path, each object it links by its path; none where it holds none, where a path above resolves to no object

In [ ]:
const path = plain({ singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book", properties: { key: "cy" },
  next: { target: {} } } } });
const document = below.document(path);
const nobody = plain({ singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book",
  properties: { key: "zed" }, next: { target: {} } } } });
assert(Json.dumps([Json.dumps(document), below.document(nobody)]) === '["{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"0\\": {\\"schema\\": [{\\"$ref\\": {\\"singleton\\": \\".\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"crm\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"Person\\"}, \\"next\\": {\\"target\\": {}}}}}}}}]}}, \\"spec\\": [{\\"0\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"cy\\"}}]}}, {\\"as_adjacency\\": {\\"name\\": \\"book\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"book\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"target\\": {}}}}}}]}}, {\\"as_property\\": {\\"name\\": \\"key\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"cy\\"}}]}}]}]}}]}}, {\\"as_adjacency\\": {\\"name\\": \\"befriended\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"host\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"bob\\"}, \\"next\\": {\\"target\\": {}}}}}}}}]}}, {\\"as_link\\": {\\"name\\": \\"guest\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"ada\\"}, \\"next\\": {\\"target\\": {}}}}}}}}]}}]}]}}]}}]}]}]}}]}", null]');
const reference = Plain.FromPlain(above)(above.registered("References.OfObject"), new Map([["path", nobody]]));
assert(References.resolve(above, reference) === null);

<a id="BLW-04"></a>
## BLW-04 · Loading is no transaction of the store's: its followers are given nothing of it, and a transaction open while an object loads keeps its own changes alone

In [ ]:
const effects: any[] = [];
const end = Changes.observe(above, (effect) => effects.push(effect));
const cy = keyed(ours.contacts, "cy").contact;
let seen: string = "";
above.transaction(() => {
  seen = cy.name.get(); // loaded within the transaction
  above.crm.Person(cy, (p: any) => p.name("Cy"));
});
end();
assert(effects.length === 1);
const [effect] = effects;
assert(Json.dumps([seen, cy.name.get(), Json.dumps(Plain.ToPlain(above).Transaction(effect))]) === '["cy", "Cy", "{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"o0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"cy\\"}, \\"next\\": {\\"target\\": {}}}}}}}}, \\"spec\\": [{\\"o0\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"Cy\\"}}]}}]}]}]}}]}"]');

<a id="BLW-05"></a>
## BLW-05 · An object loaded has its reference as the store below has it; given no store below, a store holds what is written to it

In [ ]:
const alone: any = new B.Store();
alone.transaction(crm);
alone.transaction(() => {
  alone.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("dee").book(alone.singleton("crm.TheBook")))));
});

/** An object's reference, as plain data. */
function pathOf(store: any, x: unknown): string {
  return Json.dumps(Plain.ToPlain(store)(store.registered("References.OfObject"), References.of(store, x)));
}

assert(Json.dumps([pathOf(above, cy) === pathOf(below, people.cy),
  [...alone.singleton("crm.TheBook").contacts].map((e: any) => e.key.get()), alone._below]) === '[true, ["dee"], null]');

<a id="BLW-06"></a>
## BLW-06 · An object the store above makes is found again by its path, and one the store below adds after its owner is loaded above is loaded when it is named

In [ ]:
let eve: any = null;
above.transaction(() => {
  eve = above.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("eve").book(ours))).name("eve"));
});
const found = References.resolve(above, References.of(above, eve));
below.transaction(() => {
  below.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("fay").book(book))).name("fay"));
});
const fayPath = plain({ singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book", properties: { key: "fay" },
  next: { target: {} } } } });
const fay: any = References.resolve(above, Plain.FromPlain(above)(above.registered("References.OfObject"), new Map([["path", fayPath]])));
assert(Json.dumps([found === eve, fay.name.get()]) === '[true, "fay"]');

<a id="BLW-07"></a>
## BLW-07 · A transaction above that removes the owning entry of an object not present writes it by its owner's key, loading nothing, and removes, among the objects present, every entry linking it; one that removes an entry linking an object not present, which it names, loads it; a link read within a transaction is resolved there too

In [ ]:
below.transaction(() => { // a contact the store above will remove, and one who knows it
  const gus = below.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("gus").book(book))).name("gus"));
  below.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("hal").book(book))).name("hal")
    .knows((a: any) => a.insert((e: any) => e.known(gus))));
});
const third: any = new B.Store(below);
third.transaction(crm);
const books = third.singleton("crm.TheBook");
const hal = keyed(books.contacts, "hal").contact;
const knew = BigInt([...hal.knows].length);
const effects7: any[] = [];
Changes.observe(third, (effect) => effects7.push(effect));
let within = "";
third.transaction(() => {
  third.crm.Book(books, (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key("gus")))); // gus not present
  within = keyed(books.contacts, "ada").contact.name.get(); // ada loaded, within it
});
const removal = Json.dumps(Plain.ToPlain(third).Transaction(effects7.at(-1)));
const ada3 = keyed(books.contacts, "ada").contact;
const loadedBefore = third._loaded.size;
third.transaction(() => {
  third.crm.Person(ada3, (p: any) => p.knows((a: any) => a.clear())); // ada knew bob, not present
});
const unknowing = Json.dumps(Plain.ToPlain(third).Transaction(effects7.at(-1)));
assert(Json.dumps([knew, BigInt([...hal.knows].length), within, removal, BigInt(third._loaded.size - loadedBefore),
  unknowing]) === '[1, 0, "ada", "{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"o0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"target\\": {}}}}}}, \\"spec\\": [{\\"o0\\": [{\\"as_adjacency\\": {\\"name\\": \\"contacts\\", \\"mutations\\": [{\\"remove\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_property\\": {\\"name\\": \\"key\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"gus\\"}}]}}]}]}}]}}]}]}]}}]}", 1, "{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"o0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"bob\\"}, \\"next\\": {\\"target\\": {}}}}}}}, \\"o1\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"ada\\"}, \\"next\\": {\\"target\\": {}}}}}}}}, \\"spec\\": [{\\"o1\\": [{\\"as_adjacency\\": {\\"name\\": \\"knows\\", \\"mutations\\": [{\\"remove\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"known\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"symbol\\": \\"o0\\"}}}]}}]}]}}]}}]}]}]}}]}"]');

<a id="BLW-08"></a>
## BLW-08 · Stores stack: a store above a store above is given documents in which the objects not present in the middle one are by their paths; an entry of three links, one of them read, holds the other by its path still

In [ ]:
const middle: any = new B.Store(below);
middle.transaction(crm);
const top: any = new B.Store(middle);
top.transaction(crm);
const host = keyed(top.singleton("crm.TheBook").contacts, "bob").contact;
const [introduction] = [...host.hosts];
const guest = introduction.guest.name.get();
const stillByPath = Repr.typeName(Proxies._innerOf(introduction)._links.get("friend"));
assert(Json.dumps([guest, stillByPath, introduction.friend.name.get(), loaded(middle), loaded(top)]) === '["ada", "_Unresolved", "Cy", 4, 4]');

<a id="BLW-09"></a>
## BLW-09 · A transaction above is written through: the store below commits it first, within it, and a refusal there refuses it; what another wrote below meanwhile is applied once it ends, in the order the store below committed it

In [ ]:
const effects9: any[] = [];
const end9 = Changes.observe(above, (effect) => effects9.push(effect));
above.transaction(() => { // the store below commits it first
  above.crm.Person(bob, (p: any) => p.name("Bob"));
  above.crm.Book(ours, (b: any) => b.leads((a: any) => a.insert((e: any) => e.lead(ada))));
});
const written = [people.bob.name.get(), [...book.leads].map((e: any) => e.lead.name.get()), BigInt(effects9.length)];
raises(ValueError, () => above.transaction(() => { // ada's lead given to bob above, while another gives it to cy below
  above.crm.Book(ours, (b: any) => b.leads((a: any) => a.remove((e: any) => e.lead(ada)).insert((e: any) => e.lead(bob))));
  below.transaction(() => {
    below.crm.Book(book, (b: any) => b.leads((a: any) => a.remove((e: any) => e.lead(people.ada))
      .insert((e: any) => e.lead(people.cy))));
  });
}), "breaks unique");
end9();
assert(Json.dumps([written, BigInt(effects9.length), [...ours.leads].map((e: any) => e.lead.name.get()),
  [...book.leads].map((e: any) => e.lead.name.get())]) === '[["Bob", ["ada"], 1], 2, ["Cy"], ["Cy"]]');

<a id="BLW-10"></a>
## BLW-10 · What another writes below is applied above to the objects present alone, loading nothing: an object it made, or one not present, is linked by its path; the followers above are told, and so is a store above that one; a counted object it links is found by its value, or made

In [ ]:
const effects10: any[] = [];
const end10 = Changes.observe(above, (effect) => effects10.push(effect));
const before10 = above._loaded.size;
below.transaction(() => { // ada renamed, present above; hal renamed, not present; joy made, knowing ada
  below.crm.Person(people.ada, (p: any) => p.name("Ada"));
  below.crm.Person(keyed(book.contacts, "hal").contact, (p: any) => p.name("Hal"));
  below.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("joy").book(book))).name("joy")
    .knows((a: any) => a.insert((e: any) => e.known(people.ada))));
});
end10();
const applied = effects10.map((effect) => Json.dumps(Plain.ToPlain(above).Transaction(effect)));
const loaded10 = BigInt(above._loaded.size - before10);
const [knowing] = [...ada.known_by];
const byPath = Repr.typeName(Proxies._innerOf(knowing)._links.get("knower"));
assert(Json.dumps([ada.name.get(), applied, loaded10, byPath, knowing.knower.name.get(), BigInt(above._loaded.size - before10),
  introduction.guest.name.get()]) === '["Ada", ["{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"o0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"joy\\"}, \\"next\\": {\\"target\\": {}}}}}}}, \\"o1\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"target\\": {}}}}}, \\"o2\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"ada\\"}, \\"next\\": {\\"target\\": {}}}}}}}}, \\"spec\\": [{\\"o1\\": [{\\"as_adjacency\\": {\\"name\\": \\"contacts\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"contact\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"symbol\\": \\"o0\\"}}}]}}, {\\"as_property\\": {\\"name\\": \\"key\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"joy\\"}}]}}]}]}}]}}]}, {\\"o2\\": [{\\"as_adjacency\\": {\\"name\\": \\"known_by\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"knower\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"symbol\\": \\"o0\\"}}}]}}]}]}}]}}]}, {\\"o2\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"Ada\\"}}]}}]}]}]}}]}"], 0, "_Unresolved", "joy", 1, "Ada"]');
below.transaction(() => { // joy, present above, given a tag
  below.crm.Person(keyed(book.contacts, "joy").contact,
    (p: any) => p.tags((a: any) => a.insert((e: any) => e.tag(below.crm.Tag((t: any) => t.label("vip"))))));
});
assert(Json.dumps([...knowing.knower.tags].map((e: any) => e.tag.label.get())) === '["vip"]');

<a id="BLW-11"></a>
## BLW-11 · An object another deletes below goes above with every entry linking it; an entry read after, held by no object, links what the store below still holds, and refuses a link to what it no longer holds

In [ ]:
const halEntry = keyed(ours.contacts, "hal"); // hal not present above
const [walking] = [...bob.walks]; // bob walks rex, not present above
const byPath11 = [Repr.typeName(Proxies._innerOf(halEntry)._links.get("contact")),
  Repr.typeName(Proxies._innerOf(walking)._links.get("walked"))];
below.transaction(() => { // hal removed; bob walks rex no more
  below.crm.Book(book, (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key("hal"))));
  below.crm.Person(people.bob, (p: any) => p.walks((a: any) => a.clear()));
});
raises(LookupError, () => halEntry.contact, "the store below holds no object at this path");
assert(Json.dumps([byPath11, [...ours.contacts].map((e: any) => e.key.get()).sort(), BigInt([...bob.walks].length),
  walking.walked.kind.get()]) === '[["_Unresolved", "_Unresolved"], ["ada", "bob", "cy", "eve", "fay", "joy"], 0, "dog"]');

<a id="BLW-12"></a>
## BLW-12 · An entry a transaction rewrites, linking an object not present, links it when it loads within the transaction, in the transaction's tables alone, as one loaded in a transaction that writes nothing linking it does; written through, the store below holds the entry as rewritten

In [ ]:
const fourth: any = new B.Store(below);
fourth.transaction(crm);
const ada4 = keyed(fourth.singleton("crm.TheBook").contacts, "ada").contact;

/** Ada made the friend of each introduction she is the guest of. */
function befriend(p: any): any {
  for (const e of p.guested) e.friend(ada4);
  return p;
}

let rewritten: any = null;
let byPath12 = "";
let host12 = "";
fourth.transaction(() => {
  fourth.crm.Person(ada4, befriend);
  [rewritten] = [...ada4.guested];
  byPath12 = Repr.typeName(Proxies._innerOf(rewritten)._links.get("host"));
  host12 = rewritten.host.name.get(); // bob loaded within it
});
const bob4 = rewritten.host;
let named12 = "";
fourth.transaction(() => { // one that writes nothing an object loaded within it links
  named12 = keyed(fourth.singleton("crm.TheBook").contacts, "eve").contact.name.get();
});
assert(Json.dumps([named12, byPath12, host12, [...ada4.guested].map((e: any) => [e.host.name.get(), e.friend.name.get()]),
  [...bob4.hosts].map((e: any) => [e.guest.name.get(), e.friend.name.get()]),
  [...people.ada.guested].map((e: any) => [e.host.name.get(), e.friend.name.get()])]) === '["eve", "_Unresolved", "Bob", [["Bob", "Ada"]], [["Ada", "Ada"]], [["Bob", "Ada"]]]');

<a id="BLW-13"></a>
## BLW-13 · An entry another writes below, linking an object present above whose schema has no adjacency to it by name and one not present, is refused where it is applied: name one

In [ ]:
raises(ValueError, () => below.transaction(() => { // kim, made, mentors ada, present above, who has no adjacency to it by name
  below.crm.Person((p: any) => p.book((a: any) => a.insert((e: any) => e.key("kim").book(book)))
    .mentors((a: any) => a.insert((e: any) => e.mentee(people.ada))));
}), "name one");